## For positive_heart_AB controls of 80K NGN2 derived neurons
- `positive_heart_AB`
- only information from the fasta given

In [4]:
from importlib import reload
import pandas as pd
import sys
import os
sys.path.append('../helpful_functions')
import helpful_functions as hf
reload(hf)


<module 'helpful_functions' from '/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/../helpful_functions/helpful_functions.py'>

In [5]:
# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'


interesting_columns = [col_name, col_sequence, col_category, col_class, col_source, col_ref, col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info]

In [6]:
import yaml

# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/config_file.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

input_fasta = config['design_file']
pre_metadata_df = hf.fasta_to_dataframe(input_fasta, columns=[col_name, col_sequence])
pre_metadata_df = hf.fasta_to_dataframe(input_fasta, columns=[col_name, col_sequence])
pre_metadata_df['tmp_label'] = pre_metadata_df[col_name].apply(lambda x: hf.get_label(x))


In [7]:
def get_coordinates_control_pos_heart_AB(row):
    """
    Special cases to set chr, start, end and strand for control sequences from their header (because not in region bed)
    Case: C_positive_heart_AB:
            header: C_positive_heart_AB:SKI-ENST00000378536.5:Oligo-Sub-Id:0.1:chr1:2226527-2226797:Length::270
                => chr: 1, start: 2226527, end: 2226797, strand: + (assume all are +)
    """
    name = row[col_name]
    print(name)
    if name.startswith('C_positive_heart_AB'): # >C_positive_heart_AB:SKI-ENST00000378536.5:Oligo-Sub-Id:0.1:chr1:2226527-2226797:Length::270
        region_info = name.split(':chr')[1].split(':Length')[0] # 1:2226527-2226797
        row[col_chr] = f"chr{region_info.split(':')[0]}"
        row[col_start] = int(region_info.split(':')[1].split('-')[0])
        row[col_end] = int(region_info.split(':')[1].split('-')[1])
        row[col_strand] = '.'
        row[col_class] = 'element inactive control'
        row[col_info] = row[col_info] + ';  strand information not restorable from header'
    return row

In [8]:
group_name = 'C_positive_heart_AB'
# # filter for group to get an overview
pre_metadata_df_group = pre_metadata_df.loc[pre_metadata_df['tmp_label'] == group_name].copy()
pre_metadata_df_group # 909

# add the columns of the metadata file
pre_metadata_df_group[col_sequence] = pre_metadata_df_group[col_sequence].apply(lambda x: x[15:-15])
pre_metadata_df_group[col_category] = 'element'
pre_metadata_df_group[col_class] = 'element inactive control'
pre_metadata_df_group[col_source] = 'general controls IGVF year 1 design 2023'
pre_metadata_df_group[col_ref] = 'GRCh38'
pre_metadata_df_group[col_chr] = 'NA'
pre_metadata_df_group[col_start] = 'NA'
pre_metadata_df_group[col_end] = 'NA'
pre_metadata_df_group[col_strand] = 'NA'
pre_metadata_df_group[col_variant_class] = 'NA'
pre_metadata_df_group[col_variant_pos] = 'NA'
pre_metadata_df_group[col_SPDI] = 'NA'
pre_metadata_df_group[col_allele] = 'NA'
pre_metadata_df_group[col_info] = '' # 'Coordinates are based on GRCh37 (wrong genome build)'

pre_metadata_df_group = pre_metadata_df_group.apply(get_coordinates_control_pos_heart_AB, axis=1)

C_positive_heart_AB:SKI-ENST00000378536.5:Oligo-Sub-Id:0.1:chr1:2226527-2226797:Length::270
C_positive_heart_AB:SKI-ENST00000378536.5:Oligo-Sub-Id:0.2:chr1:2226707-2226977:Length::270
C_positive_heart_AB:SKI-ENST00000378536.5:Oligo-Sub-Id:0.3:chr1:2226887-2227157:Length::270
C_positive_heart_AB:SKI-ENST00000378536.5:Oligo-Sub-Id:0.4:chr1:2227067-2227337:Length::270
C_positive_heart_AB:SKI-ENST00000378536.5:Oligo-Sub-Id:0.5:chr1:2227247-2227517:Length::270
C_positive_heart_AB:SKI-ENST00000378536.5:Oligo-Sub-Id:0.6:chr1:2227427-2227697:Length::270
C_positive_heart_AB:SKI-ENST00000378536.5:Oligo-Sub-Id:0.7:chr1:2227607-2227877:Length::270
C_positive_heart_AB:SKI-ENST00000378536.5:Oligo-Sub-Id:0.8:chr1:2227787-2228057:Length::270
C_positive_heart_AB:SKI-ENST00000378536.5:Oligo-Sub-Id:0.9:chr1:2227967-2228237:Length::270
C_positive_heart_AB:PEX10-ENST00000515760.1:Oligo-Sub-Id:0.1:chr1:2411502-2411772:Length::270
C_positive_heart_AB:PEX10-ENST00000515760.1:Oligo-Sub-Id:0.2:chr1:2411682-2411

In [10]:
pre_metadata_df_group
# check if the row sum is equal to the initial row number
print(f"Expected number of rows: {pre_metadata_df.loc[pre_metadata_df['tmp_label'] == 'C_positive_heart_AB'].shape[0]}")
print(f'Number of rows with matching coordinates: {pre_metadata_df_group.shape[0]}')

group_name = 'C_positive_heart_AB'
output_dir = config['final_output_dir']
output_path = os.path.join(output_dir, group_name)
# Write DataFrame to TSV file
pre_metadata_df_group[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')

Expected number of rows: 909
Number of rows with matching coordinates: 909


0

In [ ]:
def get_start_end_strand_control(row):
    """
    Special cases to set chr, start, end and strand for control sequences from their header (because not in region bed)
    Case: C_positive_heart_AB:
            header: C_positive_heart_AB:SKI-ENST00000378536.5:Oligo-Sub-Id:0.1:chr1:2226527-2226797:Length::270
                => chr: 1, start: 2226527, end: 2226797, strand: + (assume all are +)

    Case: MK: (all MK sequences have "|chr" pattern)
            header: MK:rdhs_664198|chr8-10365156+10365425|reference
                => chr: 8, start: 10365156, end: 10365425, strand: +
            header: MK:ACAP2|chr3-195443140-195443409|reference
                => chr: 3, start: 195443140, end: 195443409, strand: -

    Case: C_negative_neuron_MK: (all C_negative_neuron_MK sequences have "_chr" pattern)
            header: C_negative_neuron_MK:tile_14444_chr15_67066278_67066547_reference__1.1385203581298
                => chr: 15, start: 67066278, end: 67066547, strand: . (no information given)
            variant_header: >C_negative_neuron_MK:tile_36043_chr6_14500968_14501237_G_C_261__0.274168942409752

    Case: C_positive_neuron_MK: (all C_positive_neuron_MK sequences have "_chr" pattern")
            header: C_positive_neuron_MK:tile_35742_chr6_3247831_3248100_reference_0.892141141777512
                => chr: 6, start: 3247831, end: 3248100, strand: . (no information given)

    Case: C_negative_heart_MK: ( all C_negative_heart_MK sequences have "_chr" pattern)
            header: C_negative_heart_MK:tile_6903_chr11_9614045_9614314_reference__0.958461950470297
                => chr: 11, start: 9614045, end: 9614314, strand: . (no information given)

    Case: C_positive_heart_MK: (all C_positive_heart_MK sequences have "_chr" pattern)
            header: C_positive_heart_MK:tile_7939_chr11_65487592_65487861_reference_1.25449216981846
                => chr: 11, start: 65487592, end: 65487861, strand: . (no information given)

    Case: C_negative_neuron_NP: (all C_negative_neuron_NP sequences have "_chr" pattern)
            header: C_negative_neuron_NP:GW18_PFC_ABC_chr15_89400286_89400556_0.830617698776558
                => chr: 15, start: 89400286, end: 89400556, strand: . (no information given)
            additional condition: scrambled_control____2.28928058620308
                => category: scrambled

    Case: C_positive_neuron_NP: (all C_positive_neuron_NP sequences have "_chr" pattern)
            header: C_positive_neuron_NP:GW18_PFC_ABC_chr11_65487667_65487937_5.27702983385667
                => chr: 11, start: 65487667, end: 65487937, strand: . (no information given)

    Case: C_SLEA: (all C_SLEA sequences have ":chr" pattern and all have "|" as separator)
            header: C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:V_HNF4_Q6:AAGGTCCAG;155:V_HNF4_Q6:AAGGTCCAG
                => chr: 2, start: 210861483, end: 210861650, strand: . (no information given)
            additional condition: category: 'synthetic', ref: 'hg18', strand: '.'

    Case: C_positive_heart_CAD: (headers need to be split by "_")
            header: C_positive_heart_CAD:REF_rs604723
                => use region map

    Case: C_positive_neuron_CD: headers have "::chr" pattern and delimited by "-mean_ratio"
            header: C_positive_neuron_CD:p1_rs6813360_A_C_ref_50_A::chr4:155359187-155359457-mean_ratio2.42
                => chr: 4, start: 155359187, end: 155359457, strand: . (no information given)
            additional condition: C_positive_neuron_CD:c1_NA_NA_NA_NA::72hr_top_94-mean_ratio2.31

    Case:  GC_DNase_positive: header have ":chr" pattern delimited by "_active_count_" (Coordinates are based on GRCh37)
            header: GC_DNase_positive:chr9:88419918-88420187_active_count_112[.]

    Case: GC_DNase_positive_shuffeled: header have ":chr" pattern delimited by "_active_count_" (Coordinates are based on GRCh37)
            header: GC_DNase_positive_shuffeled:chr1:121484605-121484874_active_count_114
                => chr: 1, start: 121484605, end: 121484874, strand: . (no information given)
                + shuffled

    Case: GC_DNase_negative_blood_shuffeled: header have ":chr" pattern and delimited by "_active_count_" (Coordinates are based on GRCh37)
            header: GC_DNase_negative_blood_shuffeled:chr1:8213090-8213359_active_count_12_1
                => chr: 1, start: 8213090, end: 8213359, strand: . (no information given)
                + shuffled

    Case: GC_DNase_negative_brain_shuffeled: header have ":chr" pattern and delimited by "_active_count_" (Coordinates are based on GRCh37)
            header: GC_DNase_negative_brain_shuffeled:chr1:4768943-4769212_active_count_9_2
                => chr: 1, start: 4768943, end: 4769212, strand: . (no information given)
                + shuffled

    Case: GC_DNase_negative_brain: header have ":chr" pattern and delimited by "_active_count_" (Coordinates are based on GRCh37)
            header: GC_DNase_negative_brain:chr1:4768943-4769212_active_count_9_2[.]
                => chr: 1, start: 4768943, end: 4769212, strand: . (no information given)

    Case: GC_DNase_negative_blood: header have ":chr" pattern and delimited by "_active_count_" (Coordinates are based on GRCh37)
            header: GC_DNase_negative_blood:chr1:8213090-8213359_active_count_12_1[.]
                => chr: 1, start: 8213090, end: 8213359, strand: . (no information given)
    """

    if row[col_category] == 'synthetic' or row[col_category] == 'scrambled':
        return row
    name = row[col_name]
    # print(name) # TODO: remove debug
    if name.startswith('C_positive_heart_AB'): # >C_positive_heart_AB:SKI-ENST00000378536.5:Oligo-Sub-Id:0.1:chr1:2226527-2226797:Length::270
        region_info = name.split(':chr')[1].split(':Length')[0] # 1:2226527-2226797
        row[col_chr] = f"chr{region_info.split(':')[0]}"
        row[col_start] = int(region_info.split(':')[1].split('-')[0])
        row[col_end] = int(region_info.split(':')[1].split('-')[1])
        row[col_strand] = '.' # TODO: make sure which strand they are on
        row[col_class] = 'element inactive control'
        row[col_info] = row[col_info] + '; strand information not restorable from header'

    elif name.startswith('MK'):
        row[col_class] = 'element inactive control' # for the tested sequences the MK scrambled can be used as negative control; the rest of MK is of unknown importance
        if 'scramble' in name:
            row[col_category] = 'scrambled'
            return row
        region_info = name.split('|chr')[1].split('|')[0] # 8-10365156+10365425 or 3-195443140-195443409
        row[col_chr] = f"chr{region_info.split('-')[0]}"
        row[col_strand] = '+' if '+' in region_info else '-'
        genomic_range = region_info.split('-')[1]
        if row[col_strand] == '-': # if "-" strand then genomic region is already split
            genomic_range = '-'.join(region_info.split('-')[1:])
        row[col_start] = int(genomic_range.split(row[col_strand])[0])
        row[col_end] = int(genomic_range.split(row[col_strand])[1])

    elif name.startswith('C_negative_heart_MK') or name.startswith('C_negative_neuron_MK') or name.startswith('C_positive_heart_MK') or name.startswith('C_positive_neuron_MK'):
        if 'scramble' in name:
            row[col_category] = 'scrambled'
            return row

        row[col_class] = 'element inactive control'
        if 'positive_neuron' in name:
            row[col_class] = 'element active control'
        region_info = '_'.join(name.split('_chr')[1].split('_')[:3]) # 11_9614045_9614314
        row[col_category] = 'element'
        row[col_chr] = f"chr{region_info.split('_')[0]}"
        row[col_start] = int(region_info.split('_')[1]) -1 # turn into 0-based
        row[col_end] = int(region_info.split('_')[2])
        row[col_strand] = '.'
        if len(name.split('_chr')[1].split("_")) == 7:
            row[col_variant_class] = 'SNV'
            row[col_category] = 'variant'
            row[col_class] = 'variant negative control'
            row[my_col_ref_base] = name.split('_chr')[1].split("_")[3]
            row[my_col_alt_base] = name.split('_chr')[1].split("_")[4]
            row[col_variant_pos] = int(name.split('_chr')[1].split("_")[5]) - 1
            if 'positive_neuron' in name:
                row[col_class] = 'variant positive control'

    elif name.startswith('C_negative_neuron_NP') or name.startswith('C_positive_neuron_NP'):
        row[col_class] = 'element inactive control'
        if 'scramble' in name:
            row[col_category] = 'scrambled'
            return row
        if 'positive_neuron' in name:
            row[col_class] = 'element active control'
        region_info = '_'.join(name.split('_chr')[1].split('_')[:3]) # 11_9614045_9614314
        # print(region_info)
        row[col_chr] = f"chr{region_info.split('_')[0]}"
        row[col_start] = int(region_info.split('_')[1])
        row[col_end] = int(region_info.split('_')[2])
        row[col_strand] = '.'

    elif name.startswith('C_SLEA'):
        row[col_class] = 'element inactive control'
        row[col_category] = 'synthetic'
        row[col_ref] = 'hg18'
        region_info = name.split(':chr')[1].split('|')[0] # 2:210861483-210861650
        row[col_chr] = f"chr{region_info.split(':')[0]}"
        row[col_start] = int(region_info.split(':')[1].split('-')[0])
        row[col_end] = int(region_info.split(':')[1].split('-')[1])
        row[col_strand] = '.'

    elif name.startswith('C_positive_heart_CAD'):
        row[col_class] = 'element inactive control'
        print("TODO: use region map + bed file or add this information to the final variant map / bed file")

    elif name.startswith('C_positive_neuron_CD'):
        row[col_class] = 'element active control'
        if 'NA_NA_NA' in name:
            row[col_category] = 'scrambled'
            row[col_chr] = 'NA'
            row[col_start] = 'NA'
            row[col_end] = 'NA'
            row[col_strand] = 'NA'
            row[col_info] = row[col_info] + '; Information not restorable from header'
            return row
        region_info = name.split('::chr')[1].split('-mean_ratio')[0] # 4:155359187-155359457
        row[col_chr] = f"chr{region_info.split(':')[0]}"
        row[col_start] = int(region_info.split(':')[1].split('-')[0])
        row[col_end] = int(region_info.split(':')[1].split('-')[1])
        row[col_strand] = '.'

    elif name.startswith('GC_DNase_positive:') or name.startswith('GC_DNase_negative_brain:') or name.startswith('GC_DNase_negative_blood:'):
        row[col_class] = 'element inactive control'
        region_info = name.split(':chr')[1].split('_active_count_')[0]
        row[col_chr] = f"chr{region_info.split(':')[0]}"
        row[col_start] = int(region_info.split(':')[1].split('-')[0])
        row[col_end] = int(region_info.split(':')[1].split('-')[1])
        row[col_strand] = '.'
        # add info that coordinates of GRCh37 are used
        row[col_info] = row[col_info] + '; Coordinates are based on GRCh37'

    elif name.startswith('GC_DNase_positive_shuffeled:') or name.startswith('GC_DNase_negative_blood_shuffeled') or name.startswith('GC_DNase_negative_brain_shuffeled'):
        row[col_class] = 'element inactive control'
        region_info = name.split(':chr')[1].split('_active_count_')[0]
        row[col_category] = 'scrambled' # all sequences are scrambled
        row[col_chr] = f"chr{region_info.split(':')[0]}"
        row[col_start] = int(region_info.split(':')[1].split('-')[0])
        row[col_end] = int(region_info.split(':')[1].split('-')[1])
        row[col_strand] = '.'
        # add info that coordinates of GRCh37 are used
        row[col_info] = row[col_info] + '; Coordinates are based on GRCh37'
    else:
        print(f'No special case for {name}')
    return row